# VL-JEPA Training Demo

This notebook demonstrates training the VL-JEPA model on synthetic data.
For real training, replace SyntheticDataset with ProductDataset pointing to your data.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import yaml
from src.models.vl_jepa import VLJEPA, default_config
from src.data.dataset import SyntheticDataset
from src.training.trainer import Trainer

## 1. Setup Model and Data

In [ ]:
# Load config
with open('../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Create model
model_config = default_config()
model = VLJEPA(model_config)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Create synthetic dataset for demo
dataset = SyntheticDataset(num_samples=256, image_size=224)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=8, shuffle=True, num_workers=0)

print(f"Dataset size: {len(dataset)}")
print(f"Batches per epoch: {len(dataloader)}")

## 2. Training

In [ ]:
# Create trainer
train_config = {
    'learning_rate': 1.5e-4,
    'weight_decay': 0.05,
    'ema_momentum': 0.996,
    'alignment_weight': 0.5,
    'num_epochs': 3,
    'checkpoint_every': 5,
    'mixed_precision': False,
    'checkpoints_dir': '../checkpoints',
}

trainer = Trainer(model=model, train_loader=dataloader, config=train_config)
print("Trainer initialized.")

In [ ]:
# Train for a few epochs
history = trainer.train(num_epochs=3)
print("\nTraining complete!")
print(f"Final loss: {history[-1]['total_loss']:.4f}")

## 3. Visualize Training

In [ ]:
from src.evaluation.visualize import plot_training_curves

fig = plot_training_curves(history)
fig.savefig('../logs/training_curves.png', dpi=150, bbox_inches='tight')
print("Training curves saved to logs/training_curves.png")

## 4. Visualize Masking

In [ ]:
from src.training.masking import BlockMasking
from src.evaluation.visualize import visualize_masking
from PIL import Image
import torchvision.transforms as T

# Create a sample image (or load your own)
sample = dataset[0]
image_tensor = sample['image']

# Generate mask
masker = BlockMasking()
mask = masker.generate_mask(14, 14, num_targets=4, target_coverage=0.5)

# Visualize
fig = visualize_masking(image_tensor, mask, patch_size=16)
print("Masking visualization generated.")

## Next Steps

- Feed the agent research papers: `python agent.py learn --file paper.pdf`
- Evolve the model: `python agent.py evolve`
- Train on real product data: `python agent.py train --epochs 50`